# Plot Assembly

Before plots can be laid out, they have to be assembled. Arguably one of `patchwork`'s biggest selling points is that it expands on the use of `+` in ggplot2 to allow plots to be added together, creating a natural extension of the ggplot2 API. There is more to it than that though, and this tutorial walks through all the operators and functions available for plot assembly.

In [1]:
from ggplot2_py import aes, facet_wrap, geom_bar, geom_boxplot, geom_jitter, geom_point, ggplot, ggtitle, theme_minimal

import patchwork as pw
from patchwork._datasets import mtcars

df = mtcars().reset_index()

In [2]:
p1 = ggplot(df) + geom_point(aes(x='mpg', y='disp')) + ggtitle('Plot 1')
p2 = ggplot(df) + geom_boxplot(aes(x='gear', y='disp', group='gear')) + ggtitle('Plot 2')
p3 = ggplot(df) + geom_point(aes(x='hp', y='wt', colour='mpg')) + ggtitle('Plot 3')
p4 = ggplot(df) + geom_bar(aes(x='gear')) + facet_wrap('~cyl') + ggtitle('Plot 4')

## Adding plots to the patchwork

Use `+` to add plots together:

In [3]:
p1 + p2

A patchwork composed of 2 patches
- Autotagging is turned off
- Guides are kept

Other patchworks can also be added, creating a nested patchwork:

In [4]:
patch = p1 + p2
p3 + patch

A patchwork composed of 2 patches
- Autotagging is turned off
- Guides are kept

### Adding tables

> **Note on deviation**: the R version accepts a `gt_tbl` (from the `gt` package) and also a data.frame. `gt` has no Python port, so the Python `wrap_table` accepts a `pandas.DataFrame` or a `gtable_py.Gtable` directly. Passing a `gt_tbl`-shaped object raises `NotImplementedError`.

The minimal example uses a DataFrame:

In [5]:
p1 + pw.wrap_table(df[['mpg', 'disp']].head(10), panel='full', space='fixed')

A patchwork composed of 2 patches
- Autotagging is turned off
- Guides are kept

### Adding other content

Standard grid grobs can be added to your plot too:

In [6]:
from grid_py import text_grob

p1 + text_grob('Some really important text')

A patchwork composed of 2 patches
- Autotagging is turned off
- Guides are kept

Base-R graphics (`~plot(...)` formulas in R) have no Python analogue and are intentionally unsupported in this port. Use `ggplot2_py` directly instead.

The workhorse underneath adding non-ggplot objects is `wrap_elements()`, which is called implicitly. For more control, wrap the object explicitly:

In [7]:
p1 + pw.wrap_elements(full=text_grob('Header'))

A patchwork composed of 2 patches
- Autotagging is turned off
- Guides are kept

Another use case for `wrap_elements()` is when you need the first plot to not be a ggplot. Python's `+` dispatch can't reach a plain grob, so wrap it:

In [8]:
pw.wrap_elements(text_grob('Text on left side')) + p1

A patchwork composed of 2 patches
- Autotagging is turned off
- Guides are kept

### Stacking and packing

`|` places plots beside each other; `/` stacks them. Because Python's `/` binds tighter than `|`, **always parenthesize** when mixing:

In [9]:
p1 / p2

A patchwork composed of 2 patches
- Autotagging is turned off
- Guides are kept

In [10]:
p1 | p2

A patchwork composed of 2 patches
- Autotagging is turned off
- Guides are kept

In [11]:
p1 / (p2 | p3)

A patchwork composed of 2 patches
- Autotagging is turned off
- Guides are kept

### Functional assembly

When the plot list is dynamic, use `wrap_plots()`:

In [12]:
pw.wrap_plots(p1, p2, p3, p4)

A patchwork composed of 4 patches
- Autotagging is turned off
- Guides are kept

## Nesting the left-hand side

Plots are always added to the right of an existing patchwork, so to nest the left-hand side alongside a single plot use `-` (think of it as a hyphen — it keeps both sides at the same nesting level):

In [13]:
patch = p1 + p2
patch - p3

A patchwork composed of 2 patches
- Autotagging is turned off
- Guides are kept

`merge()` is an alternative that works with `/` and `|` because it doesn't have `-`'s precedence constraints. `wrap_plots()` also flattens nesting regardless of order.

## Modifying patches

The returned patchwork still exposes the most-recently-added plot, so you can continue adding ggplot2 objects to it:

In [14]:
(p1 + p2) + geom_jitter(aes(x='gear', y='disp'))

A patchwork composed of 2 patches
- Autotagging is turned off
- Guides are kept

Indexing into the patchwork lets you modify a specific sub-plot (0-based in Python):

In [15]:
patchwork_obj = p1 + p2
patchwork_obj[0] = patchwork_obj[0] + theme_minimal()
patchwork_obj

A patchwork composed of 2 patches
- Autotagging is turned off
- Guides are kept

### Modifying everything

`&` recurses into every sub-plot (including nested patchworks); `*` only touches the current level.

In [16]:
patchwork_obj = p3 / (p1 | p2)
patchwork_obj & theme_minimal()

A patchwork composed of 3 patches
- Autotagging is turned off
- Guides are kept

In [17]:
patchwork_obj * theme_minimal()

A patchwork composed of 3 patches
- Autotagging is turned off
- Guides are kept

## Next steps

See the `layout.ipynb` tutorial for fine-grained layout control and the `annotation.ipynb` tutorial for titles and auto-tagging.